# Silver quality és feltöltés

Ez a notebook a bronze OHLCV és silver/calendar adatokból előállítja a silver generated OHLCV táblát.

A folyamat:

1. Betölti a bronze és calendar adatokat Azure-ból.
2. Calendar alapján szűri az elvárt gyertyahelyeket.
3. Generált OHLC gyertyát készít.
4. Quality mezőket ad hozzá:
   - `consensus_quality`
   - `candle_quality`
   - `is_outlier`
   - `quality_status`
5. Broker ranking táblát készít.
6. Feltölti a silver adatokat Azure-ba havi és asset bontásban.

Két generálási módszer van:

- `method=0`: medián alapú OHLC
- `method=1`: automatikus preferred broker alapú OHLC

A két módszer külön Azure útvonalra kerül:

- `silver/generated_ohlcv/method_0/...`
- `silver/generated_ohlcv/method_1/...`


In [8]:
START_MONTH = "2024-01"
END_MONTH = "2024-02"
BROKERS = None
ASSETS = None
INTERVAL = "1m"

In [9]:
import importlib

import quality.data_loader
import quality.calendar_alignment
import quality.candle_generation

importlib.reload(quality.data_loader)
importlib.reload(quality.calendar_alignment)
importlib.reload(quality.candle_generation)

from quality.data_loader import load_bronze_ohlcv
from quality.calendar_alignment import load_calendar
from quality.candle_generation import build_broker_wide_table


bronze_df = load_bronze_ohlcv(
    start_month=START_MONTH,
    end_month=END_MONTH,
    brokers=BROKERS,
    assets=ASSETS,
    interval=INTERVAL,
    source="azure",
    print_progress=True,
)

calendar_df = load_calendar(
    start_month=START_MONTH,
    end_month=END_MONTH,
    assets=ASSETS,
    interval=INTERVAL,
    source="azure",
    print_progress=True,
)

quality_wide_df = build_broker_wide_table(
    bronze_df=bronze_df,
    calendar_df=calendar_df,
)

quality_wide_df.shape

reading bronze/binance/btcusd/2024/01/BTCUSDT-1m-2024-01.parquet
reading bronze/binance/btcusd/2024/02/BTCUSDT-1m-2024-02.parquet
reading bronze/dukascopy/btcusd/2024/01/BTCUSD-1m-2024-01.parquet
reading bronze/dukascopy/btcusd/2024/02/BTCUSD-1m-2024-02.parquet
reading bronze/dukascopy/dax/2024/01/DEUIDXEUR-1m-2024-01.parquet
reading bronze/dukascopy/dax/2024/02/DEUIDXEUR-1m-2024-02.parquet
reading bronze/dukascopy/eurusd/2024/01/EURUSD-1m-2024-01.parquet
reading bronze/dukascopy/eurusd/2024/02/EURUSD-1m-2024-02.parquet
reading bronze/dukascopy/us500/2024/01/USA500IDXUSD-1m-2024-01.parquet
reading bronze/dukascopy/us500/2024/02/USA500IDXUSD-1m-2024-02.parquet
reading bronze/dukascopy/xagusd/2024/01/XAGUSD-1m-2024-01.parquet
reading bronze/dukascopy/xagusd/2024/02/XAGUSD-1m-2024-02.parquet
reading bronze/dukascopy/xauusd/2024/01/XAUUSD-1m-2024-01.parquet
reading bronze/dukascopy/xauusd/2024/02/XAUUSD-1m-2024-02.parquet
reading bronze/interactive_brokers/dax/2024/01/DAX-1m-2024-01.parque

(348415, 36)

In [10]:
import quality.silver_ohlcv

importlib.reload(quality.silver_ohlcv)
importlib.reload(quality.candle_generation)

from quality.candle_generation import generate_candles_from_brokers
from quality.silver_ohlcv import (
    build_silver_ohlcv_from_generated,
    build_silver_ohlcv_summary,
)


method_0_generated_df = generate_candles_from_brokers(
    quality_wide_df,
    method=0,
)

method_0_silver_ohlcv_df = build_silver_ohlcv_from_generated(
    method_0_generated_df,
)

method_0_summary_df = build_silver_ohlcv_summary(
    method_0_silver_ohlcv_df,
)

method_0_summary_df

,asset,consensus_quality,candle_quality,is_outlier,quality_status,rows,generated_rows,start,end,avg_broker_count,max_close_diff_pct,max_abs_return_pct
0,BTCUSD,multi_source,bad,False,bad,4,4,2024-01-05 01:47:00+00:00,2024-02-28 17:25:00+00:00,2.000000,0.922055,1.795249
1,BTCUSD,multi_source,bad,True,bad,2,2,2024-01-05 01:48:00+00:00,2024-02-28 17:31:00+00:00,2.000000,0.846937,2.588428
2,BTCUSD,multi_source,good,False,good,79261,79261,2024-01-01 21:12:00+00:00,2024-02-29 23:59:00+00:00,2.000000,0.099967,1.533951
3,BTCUSD,multi_source,good,True,bad,3,3,2024-01-03 12:01:00+00:00,2024-01-09 21:15:00+00:00,2.000000,0.076114,2.613612
4,BTCUSD,multi_source,warning,False,warning,3699,3699,2024-01-02 00:15:00+00:00,2024-02-29 22:37:00+00:00,2.000000,0.279281,1.780826
5,BTCUSD,single_source,warning,False,warning,3431,3431,2024-01-01 00:00:00+00:00,2024-02-25 22:38:00+00:00,1.000000,0.000000,0.270369
6,DAX,missing,missing,False,missing,3,0,2024-01-24 08:00:00+00:00,2024-01-24 08:02:00+00:00,NaN,NaN,NaN
7,DAX,multi_source,good,False,good,21167,21167,2024-01-02 08:02:00+00:00,2024-02-29 16:37:00+00:00,2.000000,0.049971,0.365360
8,DAX,multi_source,warning,False,warning,108,108,2024-01-02 09:30:00+00:00,2024-02-29 16:39:00+00:00,2.000000,0.150304,0.102693
9,DAX,single_source,warning,False,warning,1079,1079,2024-01-02 08:00:00+00:00,2024-02-29 08:02:00+00:00,1.000000,0.000000,0.749692


In [11]:
method_1_generated_df = generate_candles_from_brokers(
    quality_wide_df,
    method=1,
)

method_1_silver_ohlcv_df = build_silver_ohlcv_from_generated(
    method_1_generated_df,
)

method_1_summary_df = build_silver_ohlcv_summary(
    method_1_silver_ohlcv_df,
)

method_1_summary_df


,asset,consensus_quality,candle_quality,is_outlier,quality_status,rows,generated_rows,start,end,avg_broker_count,max_close_diff_pct,max_abs_return_pct
0,BTCUSD,multi_source,bad,False,bad,3,3,2024-01-05 01:47:00+00:00,2024-02-02 13:30:00+00:00,2.000000,0.447296,0.846656
1,BTCUSD,multi_source,bad,True,bad,3,3,2024-01-05 01:48:00+00:00,2024-02-28 17:31:00+00:00,2.000000,0.922055,2.766382
2,BTCUSD,multi_source,good,False,good,79261,79261,2024-01-01 21:12:00+00:00,2024-02-29 23:59:00+00:00,2.000000,0.099967,1.544676
3,BTCUSD,multi_source,good,True,bad,3,3,2024-01-03 12:01:00+00:00,2024-01-09 21:15:00+00:00,2.000000,0.076114,2.554095
4,BTCUSD,multi_source,warning,False,warning,3699,3699,2024-01-02 00:15:00+00:00,2024-02-29 22:37:00+00:00,2.000000,0.279281,1.900231
5,BTCUSD,single_source,warning,False,warning,3431,3431,2024-01-01 00:00:00+00:00,2024-02-25 22:38:00+00:00,1.000000,0.000000,0.270369
6,DAX,missing,missing,False,missing,3,0,2024-01-24 08:00:00+00:00,2024-01-24 08:02:00+00:00,NaN,NaN,NaN
7,DAX,multi_source,good,False,good,21167,21167,2024-01-02 08:02:00+00:00,2024-02-29 16:37:00+00:00,2.000000,0.049971,0.374054
8,DAX,multi_source,warning,False,warning,108,108,2024-01-02 09:30:00+00:00,2024-02-29 16:39:00+00:00,2.000000,0.150304,0.205142
9,DAX,single_source,warning,False,warning,1078,1078,2024-01-02 08:00:00+00:00,2024-02-29 08:02:00+00:00,1.000000,0.000000,0.729309


In [12]:
import quality.broker_ranking

importlib.reload(quality.broker_ranking)

from quality.broker_ranking import build_broker_ranking


method_0_broker_ranking_df = build_broker_ranking(
    method_0_generated_df,
)

method_1_broker_ranking_df = build_broker_ranking(
    method_1_generated_df,
)

method_0_broker_ranking_df


,asset,broker,calendar_rows,broker_rows,missing_rows,coverage_ratio,mean_abs_diff,median_abs_diff,max_abs_diff,mean_abs_diff_pct,median_abs_diff_pct,max_abs_diff_pct
0,BTCUSD,binance,86400,86400,0,1.000000,7.967176,5.985000e+00,282.765000,0.017621,1.307016e-02,0.458912
1,BTCUSD,dukascopy,86400,82969,3431,0.960289,8.296641,6.370000e+00,282.765000,0.018349,1.390465e-02,0.458912
2,DAX,interactive_brokers,22360,22272,88,0.996064,0.706856,5.400000e-01,13.015000,0.004184,3.194957e-03,0.075096
3,DAX,dukascopy,22360,21360,1000,0.955277,0.737036,5.677500e-01,13.015000,0.004363,3.364984e-03,0.075096
4,EURUSD,saxo_bank,61575,59655,1920,0.968819,0.000007,5.000000e-06,0.000210,0.000659,4.604539e-04,0.019277
5,EURUSD,dukascopy,61575,58908,2667,0.956687,0.000003,0.000000e+00,0.000215,0.000318,0.000000e+00,0.019738
6,EURUSD,interactive_brokers,61575,54435,7140,0.884044,0.000003,0.000000e+00,0.000175,0.000304,0.000000e+00,0.016066
7,US500,interactive_brokers,56640,53995,2645,0.953302,1.184979,1.274250e+00,3.158000,0.024215,2.591531e-02,0.066322
8,US500,dukascopy,56640,52712,3928,0.930650,1.213822,1.303500e+00,3.158000,0.024805,2.637044e-02,0.066322
9,XAGUSD,saxo_bank,60720,58824,1896,0.968775,0.000559,5.000000e-05,0.018500,0.002448,2.211851e-04,0.080777


In [13]:
import importlib

import pipelines.silver_generated_ohlcv_runner

importlib.reload(pipelines.silver_generated_ohlcv_runner)

from pipelines.silver_generated_ohlcv_runner import (
    upload_silver_generated_ohlcv_with_preview,
)


silver_methods = [
    {
        "generation_method_id": 0,
        "generation_method": "median_ohlc",
        "silver_ohlcv_df": method_0_silver_ohlcv_df,
    },
    {
        "generation_method_id": 1,
        "generation_method": "preferred_broker_ohlc",
        "silver_ohlcv_df": method_1_silver_ohlcv_df,
    },
]

silver_upload_results = {}

for method_config in silver_methods:
    method_id = method_config["generation_method_id"]
    method_name = method_config["generation_method"]

    print(f"Uploading silver method {method_id}: {method_name}")

    silver_upload_results[method_id] = upload_silver_generated_ohlcv_with_preview(
        silver_ohlcv_df=method_config["silver_ohlcv_df"],
        interval=INTERVAL,
        generation_method_id=method_id,
        generation_method=method_name,
        overwrite=True,
        print_progress=True,
    )

silver_upload_results[0]["upload_results"], silver_upload_results[1]["upload_results"]


Uploading silver method 0: median_ohlc
Uploading silver generated OHLCV: BTCUSD 2024-01
Uploading silver generated OHLCV: DAX 2024-01
Uploading silver generated OHLCV: EURUSD 2024-01
Uploading silver generated OHLCV: US500 2024-01
Uploading silver generated OHLCV: XAGUSD 2024-01
Uploading silver generated OHLCV: XAUUSD 2024-01
Uploading silver generated OHLCV: BTCUSD 2024-02
Uploading silver generated OHLCV: DAX 2024-02
Uploading silver generated OHLCV: EURUSD 2024-02
Uploading silver generated OHLCV: US500 2024-02
Uploading silver generated OHLCV: XAGUSD 2024-02
Uploading silver generated OHLCV: XAUUSD 2024-02
Uploading silver method 1: preferred_broker_ohlc
Uploading silver generated OHLCV: BTCUSD 2024-01
Uploading silver generated OHLCV: DAX 2024-01
Uploading silver generated OHLCV: EURUSD 2024-01
Uploading silver generated OHLCV: US500 2024-01
Uploading silver generated OHLCV: XAGUSD 2024-01
Uploading silver generated OHLCV: XAUUSD 2024-01
Uploading silver generated OHLCV: BTCUSD 2

(      status   asset  year  month  \
 0   uploaded  BTCUSD  2024      1   
 1   uploaded     DAX  2024      1   
 2   uploaded  EURUSD  2024      1   
 3   uploaded   US500  2024      1   
 4   uploaded  XAGUSD  2024      1   
 5   uploaded  XAUUSD  2024      1   
 6   uploaded  BTCUSD  2024      2   
 7   uploaded     DAX  2024      2   
 8   uploaded  EURUSD  2024      2   
 9   uploaded   US500  2024      2   
 10  uploaded  XAGUSD  2024      2   
 11  uploaded  XAUUSD  2024      2   
 
                                           message  row_count  
 0   Silver generated OHLCV uploaded successfully.      44640  
 1   Silver generated OHLCV uploaded successfully.      11440  
 2   Silver generated OHLCV uploaded successfully.      31650  
 3   Silver generated OHLCV uploaded successfully.      29040  
 4   Silver generated OHLCV uploaded successfully.      31740  
 5   Silver generated OHLCV uploaded successfully.      31740  
 6   Silver generated OHLCV uploaded successfully.      